# 📗 LLM 평가와 생성 품질 지표: Faithfulness·Answer Relevancy·Answer Correctness

RAG 답변은 문장이 매끄러워도 검색해 온 문서에 없는 내용을 담거나 질문과 다른 이야기를 할 수 있습니다. 읽어 보고 "괜찮다"고 판단하는 것만으로는 여러 답변을 비교하거나 무엇이 잘못됐는지 찾기 어려우므로, 답변 품질을 지표로 평가합니다.

**RAGAS란**
- Retrieval Augmented Generation Assessment(검색 증강 생성 평가)입니다. 논문 「Ragas: Automated Evaluation of Retrieval Augmented Generation」([Es et al., 2023](https://arxiv.org/abs/2309.15217))에서 제안했습니다.
- RAG의 검색과 답변 생성을 평가용 LLM으로 자동 채점하는 지표 모음이자, 그 지표를 구현한 파이썬 라이브러리 `ragas`입니다.

**이 교안의 흐름**
- 평가 방법 네 가지와 RAGAS 지표 6종의 뜻을 보고, 사람이 미리 써 둔 모범 답안(정답 답변)이 필요한지에 따라 나눕니다.
- 할루시네이션(답변 오류)을 종류마다 어느 지표로 찾는지 나누고 원인을 정리합니다.
- 생성 품질 지표 3개(Faithfulness·Answer Relevancy·Answer Correctness)로 답변을 평가합니다.
- 일부러 넣은 결함마다 어느 지표가 떨어지는지 짝짓습니다.
- 실무에서 배포 전과 운영 중에 평가를 어떤 절차로 돌리는지 정리합니다.

**평가 대상**
- 유연근무 매뉴얼 질문 3개와, 사람이 미리 써 둔 답변 12개(질문마다 4개)입니다.
- 답변 12개 중 9개에는 결함을 일부러 넣었습니다.
- 점수는 평가용 LLM(답변을 만드는 모델과 따로 두고 채점에만 쓰는 LLM)이 매깁니다.

**이 단원에서 배우는 지표 6종**

RAGAS 지표는 RAG가 만든 결과를 0~1 점수로 평가하며, 높을수록 좋습니다. 지표 이름의 Context(컨텍스트)는 검색 문서를 가리킵니다.

| 교안 | 구분 | 지표 | 무엇을 평가하나 | 점수가 낮으면 |
|---|---|---|---|---|
| 01 | 생성 품질 | Faithfulness(답변 충실도) | **근거**: 답변이 검색 문서에 있는 내용만 말했는가 | 답변이 문서에 없거나 문서와 다른 내용을 말함 |
| 01 | 생성 품질 | Answer Relevancy(답변 관련성) | **주제**: 답변이 질문에 맞는 이야기인가(사실 여부는 보지 않고, 정답 답변 없이 판정) | 질문과 다른 이야기를 하거나 확답을 피함 |
| 01 | 생성 품질 | Answer Correctness(답변 정확도) | **사실**: 답변의 사실이 정답 답변과 일치하는가 | 정답 답변과 다른 사실을 말하거나 빠뜨림 |
| 02 | 검색 품질 | Context Precision(컨텍스트 정밀도) | **순서**: 정답 답변을 만드는 데 쓸모 있는 문서가 검색 순위 위쪽에 있는가 | 쓸모 있는 문서가 아래 순위로 밀림 |
| 02 | 검색 품질 | Context Recall(컨텍스트 재현율) | **빠짐**: 정답 답변에 필요한 내용을 검색 문서가 빠짐없이 담았는가 | 필요한 문서가 빠짐 |
| 02 | 검색 품질 | Context Relevancy(컨텍스트 관련성) | **엉뚱함**: 가져온 검색 문서가 질문과 관련 있는가(정답 답변 없이 판정) | 질문과 관련 없는 문서를 가져옴 |

지표는 평가 샘플 한 건의 네 칸 가운데 일부를 비교해 점수를 냅니다. ragas는 네 칸을 아래 영어 필드 이름으로 받습니다.

| 필드 | 이 교안의 용어 | 뜻 |
|---|---|---|
| `user_input` | 질문 | 사용자가 한 질문 |
| `retrieved_contexts` | 검색 문서 | 검색기가 질문으로 찾아 답변 모델에 넘긴 문서 조각 목록 |
| `response` | 답변 | RAG가 만든 답변 |
| `reference` | 정답 답변 | 사람이 근거 원문을 확인하고 미리 써 둔 모범 답안 문장 |

질문과 정답 답변을 짝지어 모은 것을 실무에서는 **골든셋**(정답셋)이라고 부릅니다. "골든셋이 있다"는 평가할 질문마다 정답 답변이 준비됐다는 뜻입니다.

<img src="images/02_rag_metric_map.png" width="1050" alt="지표 6종이 평가 샘플의 네 칸(질문 user_input, 검색 문서 retrieved_contexts, 답변 response, 정답 답변 reference) 가운데 어느 칸끼리 비교하는지 원으로 표시한 표. 생성 품질: Faithfulness(답변 충실도, 근거)는 검색 문서·답변, Answer Relevancy(답변 관련성, 주제)는 질문·답변, Answer Correctness(답변 정확도, 사실)는 답변·정답 답변. 검색 품질: Context Precision(컨텍스트 정밀도, 순서)과 Context Recall(컨텍스트 재현율, 빠짐)은 검색 문서·정답 답변, Context Relevancy(컨텍스트 관련성, 엉뚱함)는 질문·검색 문서. 정답 답변 칸에 원이 있는 지표는 골든셋이 있어야 계산한다.">

**검색 지표도 정답 문서가 아니라 정답 답변을 씁니다.** 정답 문서(`relevant_doc_ids`)는 질문의 답이 들어 있다고 사람이 정한 문서이고, 앞 단원의 Precision@K·Recall@K가 이 문서 ID로 검색 결과를 셉니다. RAGAS의 Context Precision·Context Recall은 문서 ID를 세지 않고, 평가용 LLM이 검색 문서를 읽어 정답 답변에 필요한 내용이 들어 있는지 판정합니다.

**이름은 비슷하지만 다른 것을 보는 두 쌍**
- **Answer Relevancy와 Answer Correctness:** "10시간을 넘어야 연장근로수당이 나옵니다"(정답은 8시간)는 질문에 맞는 이야기라 Answer Relevancy는 높지만, 사실이 틀려 Answer Correctness는 낮습니다.
- **Context Recall과 Context Relevancy:** 질문에 필요한 문서가 둘인데 1위 문서 하나만 가져오면, 그 문서는 질문과 관련 있어 Context Relevancy는 높지만 나머지 하나가 빠져 Context Recall은 낮습니다.

**실무에서 주로 쓰는 지표**

실무에서는 6종을 매번 모두 계산하지 않습니다. 쓸 수 있는 지표는 골든셋이 있는가로 갈립니다.

| 묶음 | 지표 | 실무에서 쓰는 때 |
|---|---|---|
| 골든셋 없이 계산(reference-free): 평가용 LLM이 질문·검색 문서·답변끼리 대조 | Faithfulness, Answer Relevancy, Context Relevancy | 운영 중 들어온 실제 질문에도 바로 씁니다. 다만 검색 문서가 옳다고 전제하므로 답이 실제로 맞는지는 알 수 없습니다 |
| 골든셋 필요(참조 기반): 평가용 LLM이 정답 답변과 대조 | Answer Correctness, Context Recall, Context Precision | 배포 전에 바꾼 설정이 이전보다 나빠지지 않았는지 골든셋으로 확인할 때 더합니다 |

- **가장 많이 쓰는 두 지표:** Faithfulness와 Answer Relevancy입니다. 정답 답변 없이 계산되므로 운영 중 답변에도 바로 쓸 수 있고, 검색 문서를 벗어난 답과 질문에서 엇나간 답을 잡습니다.
- **골든셋이 있으면 더하는 두 지표:** Answer Correctness(답의 사실이 맞는가)와 Context Recall(필요한 문서를 다 가져왔는가)입니다.
- **검색 순위를 조정할 때:** 골든셋이 있으면 Context Precision, 없으면 Context Relevancy를 더합니다.
- **ragas 기본값 주의:** ragas의 일괄 평가 함수 `evaluate()`는 지표를 지정하지 않으면 Faithfulness·Answer Relevancy·Context Precision·Context Recall 4개를 계산합니다. Answer Correctness가 빠져 있어 낡은 문서대로 틀리게 답한 경우를 찾지 못합니다. 이 단원은 Answer Correctness와 Context Relevancy를 더해 6개를 다룹니다.

이 교안은 생성 품질 지표 3개를 계산합니다. 검색 품질 지표 3개는 이 교안에서 분류만 보고, 계산은 교안 02에서 합니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="이번 교안에서 배울 것: 답변 품질을 지표로 평가하기(유연근무 매뉴얼 질문 3개, 사람이 쓴 답변 12개). 1 지표 6종의 뜻과 분류(평가 방법 4가지·검색 3개·생성 3개, 정답 답변 필요와 정답 답변 없이 가능), 2 할루시네이션의 종류와 원인(충실성 오류와 사실성 오류), 3 생성 지표 3개(Faithfulness·Answer Relevancy·Answer Correctness)로 답변 평가, 4 결함마다 떨어지는 지표(결함 답변 3종과 지표 짝짓기).">

**학습 목표** (그림의 번호와 같습니다)

1. **평가 방법 네 가지와 지표 6종(검색 3·생성 3)을 골든셋이 필요한지에 따라 나눕니다.**
2. **할루시네이션의 종류(충실성 오류·사실성 오류)와 원인을 설명합니다.**
3. **생성 품질 지표 3개(Faithfulness·Answer Relevancy·Answer Correctness)를 계산하고 해석합니다.**
4. **결함 답변 3종마다 어느 지표가 떨어지는지 짝짓습니다.**

## 1. LLM 답변을 평가하는 방법 네 가지

LLM 답변을 평가하는 방법은 문자열 일치, 임베딩 유사도, LLM 기반 평가, 사람 평가의 네 가지입니다. 방법마다 정답 답변이 있어야 쓸 수 있는지가 다르고, 그것으로 평가 전에 준비할 것이 정해집니다.

<img src="images/01_eval_methods_map.png" width="1050" alt="평가 방법 네 가지가 같은 두 답변을 어떻게 판정하는지 보여 주는 표. 정답 답변은 '하루 8시간을 넘으면 연장근로수당이 발생합니다.', 맞는 답(표현만 다름)은 '법정근로시간 8시간을 넘긴 시간에는 연장수당이 나옵니다.', 틀린 답(숫자만 바꿈)은 '10시간을 넘긴 시간에는 연장수당이 나옵니다.'. 문자열 일치(정답 답변 필요)는 0점·0점, 임베딩 유사도(정답 답변 필요)는 높음·높음(틀렸는데), LLM 기반 평가(정답 답변이 있으면 비교, 없으면 검색 문서와 대조)와 사람 평가(정답 답변 없어도 됨)는 맞음·틀림으로 판정한다.">

예시 질문은 "시차출퇴근 중 하루 8시간을 넘겨 일하면 연장근로수당을 받나요?"입니다.

1. **문자열 일치:** 답변과 정답 답변의 글자가 같은지 봅니다. 뜻이 같아도 표현이 다르면 0점입니다.
2. **임베딩 유사도:** 답변과 정답 답변을 임베딩 벡터로 바꿔 코사인 유사도를 계산합니다. 숫자 하나만 바꾼 틀린 답변도 문장이 비슷해 높은 값이 나옵니다.
3. **LLM 기반 평가:** 평가용 LLM이 답변의 사실을 정답 답변(없으면 검색 문서)과 대조해 판정합니다.
4. **사람 평가:** 가장 믿을 만하지만, 설정을 바꿀 때마다 답변 수백 개를 다시 읽을 수는 없습니다.

**그래서 이 단원은 LLM 기반 평가를 씁니다**

- **이유:** 문자열 일치는 표현만 다른 맞는 답을 0점으로, 임베딩 유사도는 숫자만 틀린 답을 높게 판정합니다. 사람 평가는 정확하지만 답변이 많으면 매번 읽을 수 없습니다. LLM 기반 평가만 답변의 사실을 대조하면서 많은 답변을 자동으로 채점합니다.
- **도구:** 맨 위에서 본 RAGAS의 지표를 씁니다. 이 단원의 지표 6종은 모두 LLM 기반이라 평가용 LLM으로 판정하고, Answer Relevancy와 Answer Correctness는 임베딩 모델도 함께 씁니다.
- **판정과 계산의 분담:** 평가용 LLM은 주장 나누기, 1·0 판정 같은 **판정**만 합니다. 비율·평균 같은 **점수 계산**은 ragas 코드가 합니다. 그래서 점수가 실행마다 달라지는 이유는 계산이 아니라 판정이 달라지기 때문입니다.
- **한계와 보완:** 호출 비용이 들고 실행마다 점수가 조금씩 달라지므로 큰 차이만 해석합니다. 점수가 이상한 답변은 사람이 원문을 열어 확인합니다.

## 2. 할루시네이션: 지표로 찾는 답변 오류

### 2-1. 오류 두 가지와 찾는 지표

할루시네이션(hallucination, 환각)은 모델이 그럴듯하지만 근거가 없거나 틀린 내용을 만들어 내는 현상으로, 지표 6종이 찾아내려는 대표적인 답변 결함입니다. 새 지표가 아니라 답변 오류의 종류이며, 종류마다 앞에서 본 지표 가운데 무엇으로 찾는지가 다릅니다.

- **충실성 오류:** 답변이 검색 문서와 어긋나는 오류입니다.
- **사실성 오류:** 답변이 검색 문서를 따랐는데도 실제 사실과 다른 오류입니다.

| 경우 | 예시 | 찾는 지표 |
|---|---|---|
| 검색 문서와 모순되는 내용을 말함 (충실성 오류) | 문서는 "근로일 사이 11시간 이상 연속 휴식"인데 "8시간 이상 쉬게 하면 됩니다" | Faithfulness |
| 검색 문서에 없는 내용을 덧붙임 (충실성 오류) | "이 휴식은 유급입니다" (유급 여부는 문서에 없음) | Faithfulness |
| 검색 문서가 낡아 실제 사실과 다름 (사실성 오류) | 개정 전 안내대로 "아동수당은 만 8세 미만" (현재 제도는 만 9세 미만) | Answer Correctness |
| 질문과 다른 제도의 문서를 그대로 따름 (검색 문제) | 시차출퇴근 질문에 선택근무 규정으로 답함 | Answer Correctness, 교안 02의 Context Relevancy |

<img src="images/03_hallucination_types.png" width="1050" alt="할루시네이션 사례 세 가지: 무엇과 어긋났나. 검색 문서(유연근무 매뉴얼) '정산기간이 1개월을 넘으면 근로일 사이에 11시간 이상 연속 휴식'에 대해 문서와 모순 '근로일 사이에 8시간 이상 쉬게 하면 됩니다.', 문서에 없는 내용 '11시간 이상 쉬게 하고, 이 휴식은 유급입니다.'는 충실성 오류(검색 문서와 어긋남). 검색 문서(개정 전 안내) '아동수당은 만 8세 미만'에 대해 문서가 낡음 '아동수당은 만 8세 미만까지 받습니다.'(현재 제도는 만 9세 미만)는 사실성 오류(문서는 따랐지만 사실과 다름).">

Faithfulness는 검색 문서가 옳다고 전제하고 답변이 그 문서를 따랐는지만 보므로, 표의 아래 두 행처럼 문서를 그대로 따른 답변은 Faithfulness가 높게 나옵니다. 이런 답변은 Answer Correctness로 찾으므로, 규정·금액처럼 자주 바뀌는 내용을 다루는 서비스는 현재 규정으로 확인한 정답 답변을 골든셋으로 준비합니다.

### 2-2. 할루시네이션이 생기는 원인

원인을 알아야 고칠 곳이 검색인지 답변 생성인지 정해집니다. 원인은 모델 쪽과 RAG 쪽으로 나눕니다.

모델 쪽 원인 세 가지는 할루시네이션 서베이 논문(여러 연구를 모아 정리한 논문, [Huang et al., 2023](https://arxiv.org/abs/2311.05232))이 정리한 원인에서 골랐습니다.

| 위치 | 원인 | 유연근무 매뉴얼 RAG에서 생기는 예 |
|---|---|---|
| 모델 | 학습 데이터에 없거나 낡은 지식 | 학습 시점 뒤에 바뀐 규정을 모른 채 학습 때 본 옛 규정으로 답합니다 |
| 모델 | 다음 단어를 확률로 고르는 생성 방식 | 검색 문서의 "연구개발 업무로 정산기간을 1개월 넘게 정하면 근로일 사이 11시간 연속 휴식"을 이어 쓰다가 조건을 빠뜨려 "선택근무를 하면 11시간 휴식을 줘야 합니다"로 씁니다 |
| 모델 | 답을 모를 때도 그럴듯하게 채우려는 경향 | **검색 문서에 답이 없는** 질문 "시차출퇴근 규정을 어기면 과태료는 얼마인가요?"에 "문서에서 확인할 수 없습니다"라고 하지 않고 없는 금액을 지어냅니다(매뉴얼에는 시차출퇴근 위반 과태료가 없습니다) |
| RAG | 검색 누락 | 11시간 휴식 조항이 있는 절을 가져오지 못합니다 |
| RAG | 엉뚱한 문서를 가져옴 | 선택근무 질문에 시차출퇴근 문서를 가져와 다른 제도의 규정으로 답합니다 |
| RAG | 낡은 문서 | 개정 전 매뉴얼을 가져옵니다 |
| RAG | 근거를 무시하고 답함 | **검색 문서에 답이 있는데도** 쓰지 않습니다. 검색 문서에 "정산기간은 1개월 이내"가 있는데 다른 제도에서 본 "최대 6개월"로 답합니다. 답변 프롬프트가 검색 문서만 근거로 쓰라고 정하지 않으면 생깁니다 |

"답을 모를 때도 그럴듯하게 채우려는 경향"과 "근거를 무시하고 답함"은 둘 다 문서에 없는 말을 하지만, **검색 문서에 답이 있었는가**로 구분합니다.

- 답이 없는데 금액을 지어냈다면: 모델이 모르는 것을 채운 것입니다.
- 답이 있는데 다른 값을 썼다면: 답변을 만들 때 검색 문서를 근거로 쓰지 않은 것입니다.

**원인에 따라 고칠 곳이 갈립니다**

- **검색 누락, 엉뚱한 문서를 가져옴:** 검색 단계의 설정(청크 나누기, 가져오는 문서 수)을 고칩니다.
- **낡은 문서:** 검색 대상 문서를 최신 안내로 바꿉니다.
- **모를 때 채움, 근거를 무시하고 답함:** 답변 프롬프트에 검색 문서만 근거로 쓰고, 문서에 답이 없으면 확인할 수 없다고 답하라는 규칙을 넣습니다.

이 가운데 검색 누락, 엉뚱한 문서를 가져옴, 근거를 무시하고 답함은 교안 02에서 검색 지표와 함께 어느 단계의 문제인지 진단합니다.

## 3. 실습 준비

### 3-1. 경로와 환경 변수

앞 단원과 같은 방식으로 경로를 정하고 `.env`에서 API 키를 읽습니다. 이 교안은 미리 써 둔 답변을 평가만 하므로 답변 모델과 검색기는 준비하지 않습니다.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 저장소 최상위 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

### 3-2. 평가용 LLM

- ragas 0.4의 지표는 `ragas.metrics.collections`에 있고, `await 지표.ascore(...)`로 답변 하나를 평가합니다. 결과 객체의 `.value`가 점수입니다. 4-2에서 답변 하나를 먼저 평가해 본 뒤, 여러 답변은 모아서 동시에 평가합니다.
- `ascore`는 비동기 함수지만, 주피터 셀에서는 `await`를 바로 쓸 수 있어 `asyncio.run`이 필요 없습니다.
- 일괄 평가 함수 `evaluate()`는 ragas 0.4.3에서 사용 중단 예정이라, 이 교안은 지표마다 `ascore`를 부릅니다.
- ragas의 지시문은 영어라 한국어 답변을 평가하면서도 주장·질문을 영어로 만들 때가 있습니다. 그래서 평가용 LLM에 `system_prompt`로 한국어 규칙을 넣습니다.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory

os.environ["RAGAS_DO_NOT_TRACK"] = "true"  # ragas가 익명 사용 통계를 외부 서버로 보내지 않게 합니다

# 평가용 LLM: ragas 0.4 지표는 LangChain 모델 대신 OpenAI SDK의 비동기 클라이언트를 받습니다
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-6-luna")  # gpt-5.6-luna처럼 버전에 소수점이 든 이름은 ragas 0.4.3에서 요청 오류가 납니다
eval_client = AsyncOpenAI()

KOREAN_RULE = "모든 출력 문장(질문·주장·이유)은 한국어로 작성하세요."

evaluator_llm = llm_factory(
    eval_model_name,
    client=eval_client,
    max_tokens=8192,            # 응답 길이 상한. 추론 모델은 생각하는 데 쓴 토큰도 여기에 들어가므로 기본값(1024)보다 크게 둡니다
    system_prompt=KOREAN_RULE,  # 이 평가용 LLM을 쓰는 모든 지표에 한국어 규칙이 적용됩니다
)

# Answer Relevancy·Answer Correctness가 문장 사이 의미 유사도를 계산할 때 쓰는 임베딩 모델
evaluator_embeddings = embedding_factory(
    "openai",
    model="text-embedding-3-large",  # ragas 0.4.3 임베딩은 차원 인자를 받지 않아 기본 3072차원을 씁니다
    client=eval_client,
)

print("평가용 LLM:", eval_model_name)

### 3-3. 평가 샘플 읽기

이 교안은 RAG를 실행하지 않습니다. RAG를 실행했다면 남았을 기록(답변과 그때 받은 검색 문서)을 사람이 미리 써서 파일에 넣어 두었습니다. 결함을 정확히 넣어야 결함마다 어느 지표가 떨어지는지 비교할 수 있기 때문입니다. 교안 02에서는 RAG를 직접 실행해 이 기록을 만듭니다.

`data/본실습/answer_samples.json`의 구성:

| 구분 | 키 | 내용 |
|---|---|---|
| 골든셋 | `user_input`, `reference` | 유연근무 매뉴얼 질문 3개(s1~s3)와 질문마다 정답 답변 하나 |
| 미리 저장한 RAG 실행 기록 | `responses` 안의 `response`, `retrieved_contexts`, `context_doc_ids` | 질문마다 답변 변형 4개와, 각 답변이 받은 검색 문서의 본문과 ID |

`근거대로`를 뺀 변형 3개는 일부러 결함을 넣은 답변입니다. 바꾼 내용은 `defect` 필드에 있습니다.

| 변형 | 일부러 넣은 결함 |
|---|---|
| `근거대로` | 결함 없음. 검색 문서 내용만으로 답한, 근거에 충실한 답변입니다. 본실습 검색 문서는 현재 매뉴얼이라 정답 답변과도 내용이 같아, 다른 변형과 비교하는 기준으로 씁니다 |
| `근거에_없는_내용_추가` | `근거대로` 답변에 검색 문서에 없는 그럴듯한 주장 두 개를 덧붙였습니다 |
| `숫자_조건_바꿈` | 문서의 숫자·조건을 다른 값으로 바꿨습니다 |
| `질문과_다른_이야기` | 검색 문서는 질문에 맞는 문서지만, 답변이 그 문서에서 질문과 상관없는 부분을 골라 답했습니다 |

지표마다 같은 질문의 `근거대로` 점수를 기준으로 두고, 결함을 넣은 변형에서 점수가 떨어지는지 봅니다.

In [ ]:
from pprint import pprint

# 질문 3개를 읽습니다
samples = json.loads((data_dir / "본실습" / "answer_samples.json").read_text(encoding="utf-8"))

# 첫 질문 하나를 그대로 출력해 데이터 모양을 봅니다 (질문·정답 답변·답변 변형 4개와 각 검색 문서)
pprint(samples[0], sort_dicts=False, width=110)

In [ ]:
# 첫 답변에 저장된 검색 문서를 따로 봅니다 (RAG가 이 문서를 받고 답했다고 가정한 기록)
first = samples[0]["responses"][0]
print("변형:", first["variant"], "| 검색 문서 ID:", first["context_doc_ids"])
print("검색 문서 본문 앞부분:", first["retrieved_contexts"][0][:150])

In [ ]:
# 답변 변형 12개를 한 표로 봅니다
rows = []
for sample in samples:
    for item in sample["responses"]:
        rows.append({
            "질문 ID": sample["question_id"],
            "변형": item["variant"],
            "결함": item["defect"],
            "답변": item["response"],
            "검색 문서 ID": ", ".join(item["context_doc_ids"]),  # 답변마다 미리 저장된 검색 문서
        })
display(pd.DataFrame(rows))

## 4. Faithfulness(답변 충실도): 답변의 주장이 검색 문서에 있는가

### 4-1. 주장 나누기, 검색 문서 대조, 비율 계산

Faithfulness는 답변의 주장 가운데 검색 문서로 뒷받침되는 주장의 비율입니다.

- **입력:** 질문(`user_input`)·답변(`response`)·검색 문서(`retrieved_contexts`). 정답 답변은 쓰지 않습니다.
- **계산:** 평가용 LLM이 답변을 주장으로 나누고, 주장마다 검색 문서로 뒷받침되는지 판정합니다.
- **공식:** Faithfulness = 검색 문서로 뒷받침되는 주장 수 ÷ 답변의 전체 주장 수

<img src="images/04_faithfulness_steps.png" width="1050" alt="Faithfulness(답변 충실도) 계산 세 단계. 1 주장으로 나누기: 답변 '연구개발 업무에만 1개월을 넘는 정산기간을 둘 수 있고, 최대 6개월이며, 근로일 사이에 8시간 이상 쉬게 해야 합니다.'를 ① 연구개발 업무에만 가능, ② 최대 6개월, ③ 근로일 사이 8시간 이상 휴식으로 나눈다. 2 검색 문서와 대조: 검색 문서(연구개발 업무, 최대 3개월, 근로일 사이 11시간 이상 연속 휴식)와 비교해 ①은 뒷받침됨, ②③은 뒷받침 안 됨. 3 비율 계산: 뒷받침된 주장 1개 ÷ 전체 주장 3개 = 0.33.">

s1 질문의 `숫자_조건_바꿈` 답변을 줄여 세 단계를 따라가 봅니다.

1. **주장으로 나누기:** 평가용 LLM이 답변을 "연구개발 업무에만 1개월을 넘는 정산기간을 둘 수 있다", "정산기간은 최대 6개월이다", "근로일 사이에 8시간 이상 쉬게 해야 한다"의 세 주장으로 나눕니다.
2. **검색 문서와 대조:** 주장마다 검색 문서에서 바로 끌어낼 수 있으면 1, 없으면 0으로 판정합니다. 검색 문서에는 최대 3개월, 11시간으로 적혀 있으므로 둘째·셋째 주장은 0입니다.
3. **비율 계산:** 주장 3개 중 1개만 뒷받침되므로 1 ÷ 3, 약 0.33입니다. 주장을 몇 개로 나눌지는 평가용 LLM이 정하므로 실제 주장 수는 달라질 수 있습니다.

### 4-2. 답변 변형 12개 평가

답변 하나에 평가용 LLM을 2번(주장 나누기, 검색 문서 대조) 호출하므로 12개면 24번입니다.

먼저 4-1에서 따라간 s1 질문의 `숫자_조건_바꿈` 답변 하나를 평가합니다. `ascore`에 질문·답변·검색 문서를 넘기고 `await`로 결과를 기다립니다.

In [ ]:
from ragas.metrics.collections import Faithfulness

faithfulness = Faithfulness(llm=evaluator_llm)  # 주장 나누기와 검색 문서 대조에 쓸 평가용 LLM

# 답변 하나만 평가합니다: s1 질문의 숫자_조건_바꿈 답변
sample = samples[0]            # s1 질문
item = sample["responses"][2]  # 변형 4개 가운데 셋째가 숫자_조건_바꿈
result = await faithfulness.ascore(
    user_input=sample["user_input"],
    response=item["response"],
    retrieved_contexts=item["retrieved_contexts"],  # 검색 문서는 답변 변형마다 붙어 있습니다
)
print("질문 ID:", sample["question_id"], "| 변형:", item["variant"], "| Faithfulness:", round(result.value, 3))

12개를 이렇게 하나씩 기다리면 앞 답변이 끝나야 다음 답변을 시작합니다. 그래서 12개는 동시에 평가합니다.

- **작업 만들기:** `ascore(...)`를 `await` 없이 부르면 아직 실행하지 않은 작업이 됩니다. 12개를 먼저 목록에 담아 둡니다.
- **한꺼번에 실행:** `tqdm_asyncio.gather`가 작업들을 동시에 실행하고, 넘긴 순서대로 결과 목록을 돌려줍니다. 진행 막대로 끝난 개수를 보여 줍니다.
- **동시 실행 수 제한:** 한꺼번에 너무 많은 요청을 보내면 API 요청 한도를 넘어 429 오류가 납니다. 세마포어(`asyncio.Semaphore`)로 동시에 실행되는 작업을 `MAX_CONCURRENCY`개로 묶고, 작업마다 `run_limited`로 감싸 넘깁니다.

In [ ]:
import asyncio

from tqdm.asyncio import tqdm_asyncio

# 동시에 실행할 평가 작업 수의 상한입니다. gather는 넘긴 작업을 한꺼번에 시작하므로, 작업이 많으면 API 요청 한도(분당 요청·토큰 수)를 넘어 429 오류가 날 수 있습니다.
# 32는 ragas evaluate()의 기본 동시 실행 수(RunConfig의 max_workers, 16)보다 넉넉하게 잡은 값입니다. 429 오류가 나면 줄이고, 키 한도가 넉넉하면 늘립니다.
MAX_CONCURRENCY = 32
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)  # 동시에 실행 중인 작업이 MAX_CONCURRENCY개가 되면 나머지는 자리가 날 때까지 기다립니다


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행하고 결과를 돌려줍니다."""
    async with semaphore:
        return await task

In [ ]:
# 위에서 만든 faithfulness로 답변 변형 12개의 평가를 기다리지 않고 모두 만들어 둡니다 (평가용 LLM을 답변마다 2번 호출)
pairs = []
tasks = []
for sample in samples:
    for item in sample["responses"]:
        pairs.append((sample, item))  # 결과를 어느 질문·변형에 붙일지 같은 순서로 기억합니다
        tasks.append(faithfulness.ascore(
            user_input=sample["user_input"],
            response=item["response"],
            retrieved_contexts=item["retrieved_contexts"],  # 검색 문서는 답변 변형마다 붙어 있습니다
        ))

# 동시에 MAX_CONCURRENCY개까지 실행하고, 모두 끝나면 결과를 pairs 순서대로 받습니다
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Faithfulness")

# 답변마다 점수 한 행을 쌓습니다
faithfulness_rows = []
for (sample, item), result in zip(pairs, results):
    faithfulness_rows.append({
        "질문 ID": sample["question_id"],
        "변형": item["variant"],
        "지표": "Faithfulness",
        "점수": result.value,
        "답변": item["response"],
    })
    print("평가:", sample["question_id"], item["variant"], "| 점수:", round(result.value, 3))

In [ ]:
# 행은 변형, 열은 질문 ID인 표로 바꿔 같은 질문 안에서 변형끼리 비교합니다
faithfulness_table = pd.DataFrame(faithfulness_rows).pivot(index="변형", columns="질문 ID", values="점수").round(2)
display(faithfulness_table)

`근거대로`보다 크게 낮은 변형은 검색 문서와 어긋난 주장이 있는 답변으로 보고, 원문을 검색 문서와 나란히 놓아 어느 주장인지 찾습니다.

**점수가 NaN이나 0인 행은 따로 확인합니다**

- **NaN(값 없음):** 답변에서 주장이 하나도 나오지 않으면 비율을 계산할 수 없어 NaN이 됩니다. pandas 평균은 NaN을 빼고 계산하므로, 그 칸은 질문 3개가 아니라 남은 질문만의 평균이 됩니다.
- **0:** 뽑은 주장이 모두 검색 문서로 뒷받침되지 않으면 0입니다. "확인할 수 없습니다" 같은 답도 주장 하나로 뽑히면 0이 될 수 있습니다.
- **확인만 하고 지우지 않습니다:** 아래 셀로 답변 원문을 읽어, 0이 근거 없는 주장 때문인지 회피 문장이 주장으로 뽑힌 탓인지 가립니다. 근거 없는 주장 때문이면 지표가 결함을 제대로 잡은 것입니다. 0인 행을 지우면 결함 변형의 평균이 올라가 7절 표에서 하락이 보이지 않습니다.

In [ ]:
# Faithfulness가 NaN이거나 0인 행만 골라 답변 원문과 함께 봅니다
faithfulness_df = pd.DataFrame(faithfulness_rows)
display(faithfulness_df[faithfulness_df["점수"].isna() | (faithfulness_df["점수"] == 0)])

## 5. Answer Relevancy(답변 관련성): 답변이 질문에 맞는가

### 5-1. 답변에서 질문을 거꾸로 만들기

Answer Relevancy는 답변에서 거꾸로 만든 질문이 원래 질문과 얼마나 가까운지로, 답변이 질문에 맞는지 평가합니다.

- **입력:** 질문(`user_input`)과 답변(`response`) 둘뿐입니다.
- **질문 거꾸로 만들기:** 평가용 LLM이 답변만 보고 "이 답변은 어떤 질문에 대한 답일까"를 추측해 질문을 만듭니다. 이 일을 3번 반복합니다.
- **유사도 평균:** 거꾸로 만든 질문 3개와 원래 질문의 임베딩 코사인 유사도를 구해 평균합니다.
- **임베딩 유사도 평가와의 차이:** 앞에서 본 임베딩 유사도 평가는 답변과 정답 답변을 비교했지만, 이 지표는 질문끼리 비교합니다.

<img src="images/05_answer_relevancy.png" width="1050" alt="Answer Relevancy(답변 관련성) 계산. 원래 질문 '시차출퇴근 중 더 일하면 연장·야간근로수당은 언제 받나요?'. 답변 '8시간을 넘으면 연장근로수당, 밤 10시부터 오전 6시는 야간근로수당이 나옵니다.'에서 거꾸로 만든 질문 3개(연장근로수당은 언제 나오나요?, 야간근로수당은 몇 시에 받나요?, 8시간을 넘겨 일하면 수당이 있나요?)는 의미가 가까워 점수가 높다. 답변 '시차출퇴근을 도입하면 취업규칙을 변경해 신고해야 합니다.'에서 거꾸로 만든 질문 3개(시차출퇴근 도입 시 취업규칙은?, 취업규칙 변경 신고가 필요한가요?, 시차출퇴근은 어떻게 도입하나요?)는 의미가 멀어 점수가 낮다. 점수는 거꾸로 만든 질문 3개와 원래 질문의 임베딩 유사도 평균.">

s2 질문의 `근거대로` 답변과 `질문과_다른_이야기` 답변을 줄여 따라가 봅니다.

1. s2 질문은 시차출퇴근 중 정해진 시간보다 더 일했을 때 연장근로수당과 야간근로수당을 언제 받는지 묻습니다.
2. `근거대로` 답변은 수당 조건(8시간 초과, 밤 10시부터 오전 6시)을 답합니다. 평가용 LLM이 만든 질문도 수당을 물어 원래 질문과 가깝습니다.
3. `질문과_다른_이야기` 답변은 취업규칙 변경 절차를 설명합니다. 평가용 LLM이 "시차출퇴근 도입 시 취업규칙은?" 같은 질문을 만들면, 이 질문은 원래 질문과 멀어 점수가 낮아집니다.

**회피성 답변(noncommittal)**: "상황에 따라 다릅니다"처럼 확답을 피하는 답변입니다. 평가용 LLM은 질문을 만들 때 회피성인지도 표시하고, 세 번 모두 회피성이면 점수는 0입니다.

이 지표는 답변 내용이 사실인지 확인하지 않습니다. 숫자가 틀려도 질문에 맞는 이야기라면 점수가 떨어지지 않습니다.

### 5-2. 답변 변형 12개 평가

답변 하나에 평가용 LLM을 3번(질문 만들기), 임베딩 모델을 2번(원래 질문, 만든 질문) 호출합니다. 4-2처럼 12개를 동시에 평가합니다.

In [ ]:
from ragas.metrics.collections import AnswerRelevancy

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,                # 답변에서 질문을 거꾸로 만들 평가용 LLM
    embeddings=evaluator_embeddings,  # 원래 질문과 만든 질문의 유사도를 계산할 임베딩 모델
)

# 답변 변형 12개의 평가를 모두 만들어 둡니다 (검색 문서는 이 지표에 쓰지 않습니다)
pairs = []
tasks = []
for sample in samples:
    for item in sample["responses"]:
        pairs.append((sample, item))
        tasks.append(answer_relevancy.ascore(
            user_input=sample["user_input"],
            response=item["response"],
        ))

# 4-2와 같이 동시에 MAX_CONCURRENCY개까지 실행합니다
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Answer Relevancy")

relevancy_rows = []
for (sample, item), result in zip(pairs, results):
    relevancy_rows.append({
        "질문 ID": sample["question_id"],
        "변형": item["variant"],
        "지표": "Answer Relevancy",
        "점수": result.value,
    })
    print("평가:", sample["question_id"], item["variant"], "| 점수:", round(result.value, 3))

In [ ]:
# Faithfulness 점수표와 같은 모양으로 바꿔 같은 질문 열 안에서 변형끼리 비교합니다
relevancy_table = pd.DataFrame(relevancy_rows).pivot(index="변형", columns="질문 ID", values="점수").round(2)
display(relevancy_table)

이 지표는 엇나간 답변도 0 가까이 떨어지지 않을 수 있으므로 절댓값으로 합격선을 정하지 않습니다. `근거대로`보다 뚜렷이 낮은 변형을 질문과 엇나간 답변으로 보고, 질문의 어느 부분을 놓쳤는지 원문에서 확인합니다.

### 5-3. Faithfulness와 Answer Relevancy 함께 읽기

두 지표를 함께 보면 답변을 네 가지 경우로 나눌 수 있습니다.

| | 질문에 맞음(Answer Relevancy 높음) | 질문과 엇나감(Answer Relevancy 낮음) |
|---|---|---|
| **검색 문서에 충실**(Faithfulness 높음) | `근거대로` | `질문과_다른_이야기` |
| **검색 문서를 벗어남**(Faithfulness 낮음) | `근거에_없는_내용_추가`: 가장 위험합니다(좋은 답처럼 보임) | 검색 문서를 벗어나고 질문과도 엇나간 답변 |

운영 중 답변은 두 지표를 함께 계산해 어느 칸에 드는지로 나눕니다. Answer Relevancy가 높고 Faithfulness가 낮은 답변은 사용자가 믿기 쉬우므로 다른 칸보다 먼저 확인합니다.

**확인 퀴즈**: 사용자 불만이 "답이 틀렸다"가 아니라 "물어본 것과 다른 답을 한다"라면 먼저 어느 지표를 봐야 할까요?

<details><summary>정답 보기</summary>

Answer Relevancy입니다. 답변이 질문에서 얼마나 벗어났는지를 평가하는 지표입니다. Faithfulness는 검색 문서에 충실하기만 하면 질문과 엇나간 답변에도 높게 나옵니다.

</details>

## 6. Answer Correctness(답변 정확도): 정답 답변과 맞는가

### 6-1. 주장 분류와 의미 유사도

Answer Correctness는 답변의 사실이 정답 답변과 일치하는지 평가합니다.

- **입력:** 질문(`user_input`)·답변(`response`)·정답 답변(`reference`). 검색 문서는 쓰지 않습니다.
- **공식:** Answer Correctness = 0.75 × 사실 점수 + 0.25 × 의미 유사도
- **사실 점수:** 답변의 주장과 정답 답변의 주장을 맞힌 주장(TP), 틀리거나 덧붙인 주장(FP), 빠뜨린 주장(FN)으로 나눠 구한 F1입니다. F1 = TP ÷ (TP + 0.5 × (FP + FN))이고, 분류 평가에서 쓴 2TP ÷ (2TP + FP + FN)과 같은 식입니다.
- **의미 유사도:** 답변과 정답 답변의 임베딩 코사인 유사도입니다.

<img src="images/06_answer_correctness.png" width="1050" alt="Answer Correctness(답변 정확도) 계산. 정답 답변(8시간을 넘으면 연장근로수당, 22시부터 6시는 야간근로수당, 사전 신청·승인 절차를 정해 둠)과 답변 '8시간을 넘으면 연장근로수당이 나오고, 다음 달 급여에 포함됩니다.'를 비교해 맞힌 주장(TP) 1개(8시간을 넘으면 연장근로수당), 틀리거나 덧붙인 주장(FP) 1개(다음 달 급여에 포함), 빠뜨린 주장(FN) 2개(22시부터 6시는 야간근로수당, 사전 신청·승인 절차)로 나눈다. 사실 점수(F1) = 1 ÷ (1 + 0.5 × (1 + 2)) = 0.4, 최종 점수 = 0.75 × 사실 점수 + 0.25 × 의미 유사도.">

맞힌 주장이 하나도 없는 경우를 s3 질문의 `숫자_조건_바꿈` 답변으로 따라가 봅니다.

1. **정답 답변 주장:** "9인 이하 사업장은 취업규칙 작성·신고 의무가 없다", "취업규칙이 없으면 취업규칙에 준하는 것에 규정한다", "취업규칙에 준하는 것은 서면으로 작성해 근로자에게 알린다".
2. **답변 주장:** "5인 이하 사업장만 취업규칙 없이 도입할 수 있다", "9명이면 취업규칙을 작성해 신고해야 한다".
3. **분류:** 평가용 LLM이 답변 주장 2개는 정답 답변과 어긋난다고 보아 FP로, 정답 답변 주장 3개는 답변에 없다고 보아 FN으로 분류했다고 해 봅시다. TP가 0개이므로 사실 점수는 0입니다.
4. **최종 점수:** 사실 점수가 0이면 최종 점수에는 0.25 × 의미 유사도만 남습니다.

- **주장 수는 실행마다 다를 수 있습니다:** 주장을 몇 개로 나누고 어떻게 분류할지는 평가용 LLM이 정합니다.
- **주장이 없으면 사실 점수는 1:** 답변이나 정답 답변에서 주장이 하나도 나오지 않으면 사실 점수를 1로 둡니다. NaN을 돌려주는 Faithfulness와 달리 높은 점수가 나올 수 있으므로, 높은 점수도 원문을 열어 확인합니다.

### 6-2. 답변 변형 12개 평가

답변 하나에 평가용 LLM을 3번(답변 주장 나누기, 정답 답변 주장 나누기, TP·FP·FN 분류), 임베딩 모델을 2번(답변, 정답 답변) 호출합니다. 4-2처럼 12개를 동시에 평가합니다.

In [ ]:
from ragas.metrics.collections import AnswerCorrectness

answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,                # 주장을 나누고 TP·FP·FN으로 분류할 평가용 LLM
    embeddings=evaluator_embeddings,  # 답변과 정답 답변의 의미 유사도를 계산할 임베딩 모델
)

# 답변 변형 12개의 평가를 모두 만들어 둡니다 (검색 문서는 이 지표에 쓰지 않습니다)
pairs = []
tasks = []
for sample in samples:
    for item in sample["responses"]:
        pairs.append((sample, item))
        tasks.append(answer_correctness.ascore(
            user_input=sample["user_input"],
            response=item["response"],
            reference=sample["reference"],  # 정답 답변은 질문마다 하나라 sample에서 꺼냅니다
        ))

# 4-2와 같이 동시에 MAX_CONCURRENCY개까지 실행합니다
results = await tqdm_asyncio.gather(*[run_limited(task) for task in tasks], desc="Answer Correctness")

correctness_rows = []
for (sample, item), result in zip(pairs, results):
    correctness_rows.append({
        "질문 ID": sample["question_id"],
        "변형": item["variant"],
        "지표": "Answer Correctness",
        "점수": result.value,
    })
    print("평가:", sample["question_id"], item["variant"], "| 점수:", round(result.value, 3))

In [ ]:
# Faithfulness 점수표와 같은 모양으로 바꿉니다. 의미 유사도가 1보다 작아 근거대로 답변도 1.0이 아닐 수 있습니다
correctness_table = pd.DataFrame(correctness_rows).pivot(index="변형", columns="질문 ID", values="점수").round(2)
display(correctness_table)

### 6-3. Faithfulness와 Answer Correctness 함께 읽기

두 지표를 함께 보면 답변을 네 가지 경우로 나눌 수 있습니다.

| | 정답 답변과 일치(Answer Correctness 높음) | 정답 답변과 다름(Answer Correctness 낮음) |
|---|---|---|
| **검색 문서에 충실**(Faithfulness 높음) | `근거대로` | `질문과_다른_이야기`: 답변이 받은 문서대로 말했지만 질문이 묻는 내용이 아닙니다. 검색 문서가 질문과 다른 제도이거나 낡은 경우도 이 칸에 듭니다 |
| **검색 문서를 벗어남**(Faithfulness 낮음) | `근거에_없는_내용_추가`: 정답 답변의 핵심을 담고 있어 Answer Correctness만 보면 놓칩니다 | `숫자_조건_바꿈`: 생성 단계에서 문서의 숫자·조건을 바꿨으므로 프롬프트와 모델을 확인합니다 |

Answer Correctness는 답변이 틀렸는지만 알려 주고, 왜 틀렸는지는 Faithfulness와 함께 봐야 가립니다.

- **두 지표가 함께 낮으면:** 답변이 받은 문서의 내용을 바꿔 말했으므로 답변 생성 단계(프롬프트·모델)를 고칩니다.
- **Answer Correctness만 낮으면:** 답변은 받은 문서대로 말했습니다. Answer Relevancy가 함께 낮으면 답변이 질문의 어느 부분을 놓쳤는지 보고, 받은 문서가 질문과 맞지 않거나 낡았는지는 교안 02의 검색 지표로 확인합니다.

## 7. 결함과 떨어지는 지표 짝짓기

지금까지 계산한 세 지표를 변형별로 모아, 결함마다 어느 지표가 떨어지는지 한 표로 봅니다. 칸마다 질문 3개의 점수가 들어가므로 `pivot` 대신 평균을 내는 `pivot_table`을 씁니다.

In [ ]:
# 세 지표의 행을 한 목록으로 이어 붙입니다
all_rows = faithfulness_rows + relevancy_rows + correctness_rows

# 변형과 지표의 조합마다 질문 3개의 점수를 평균합니다
summary = pd.DataFrame(all_rows).pivot_table(index="변형", columns="지표", values="점수", aggfunc="mean").round(2)
display(summary)

**확인 퀴즈**: 지표를 하나만 남긴다면 Answer Relevancy나 Answer Correctness 하나로 충분할까요? `summary`에서 그 지표가 놓치는 변형을 찾아보세요.

<details><summary>정답 보기</summary>

충분하지 않습니다.

- **`근거에_없는_내용_추가`:** 질문에 맞고 정답 답변의 내용도 모두 들어 있어, Answer Relevancy는 `근거대로`와 비슷하고 Answer Correctness도 조금 낮을 뿐입니다. 이 결함에서 점수가 크게 떨어지는 지표는 Faithfulness뿐입니다.
- **Faithfulness만 남기면:** `질문과_다른_이야기`를 놓칩니다. 받은 검색 문서에는 충실하기 때문입니다.

결함마다 떨어지는 지표를 정리하면 다음과 같습니다.

| 결함(변형) | 크게 떨어지는 지표 | 거의 그대로인 지표 |
|---|---|---|
| `근거에_없는_내용_추가` | Faithfulness | Answer Relevancy, Answer Correctness(조금 낮음) |
| `숫자_조건_바꿈` | Faithfulness, Answer Correctness | Answer Relevancy |
| `질문과_다른_이야기` | Answer Relevancy, Answer Correctness | Faithfulness |

</details>

정답의 짝짓기 표는 결함 종류를 모르는 새 답변을 점검할 때 거꾸로 씁니다. 어느 지표가 떨어졌는지로 결함 종류를 좁히고, 그 답변의 원문을 열어 확인합니다.

## 8. 실무에서 RAG를 평가하는 절차

이 교안은 질문·정답 답변·답변을 모두 사람이 미리 만든 데이터로 지표를 연습했습니다. 실무에서는 실제 질문을 RAG에 넣어 나온 답변과 그때 검색된 문서를 함께 저장하고, 그 기록으로 평가합니다. 평가는 배포 전과 운영 중 두 갈래로 돌고, 두 갈래는 골든셋으로 이어집니다.

<img src="images/07_eval_workflow.png" width="1050" alt="실무에서 RAG를 평가하는 절차. 배포 전 평가: 골든셋 만들기(실제 질문과 정답 답변), RAG 실행과 기록(답변과 검색 문서 저장), 지표 계산(질문별 점수와 평균), 기준선 비교와 원문 확인(나빠졌으면 배포 보류). 배포 뒤 운영 중 평가: 로그 수집(실제 질문·답변·검색 문서), 골든셋 없이 점검(Faithfulness·Answer Relevancy), 검토하고 골든셋 보강(낮은 점수 답변을 사람이 확인). 보강한 골든셋은 다음 배포 전 평가에 반영된다.">

### 8-1. 배포 전 평가: 골든셋으로 설정 비교

1. **골든셋 만들기:** 실제 사용자 질문이나 예상 질문을 모으고, 담당자가 근거 원문을 확인해 질문마다 정답 답변을 씁니다. 자주 묻는 질문과 틀리면 위험한 질문(금액·기한·자격)을 골고루 담습니다.
2. **RAG 실행과 기록:** 골든셋 질문을 지금의 RAG에 넣고, 나온 답변과 그때 검색된 문서를 질문 ID와 함께 저장합니다. 이 기록이 평가 샘플의 네 칸(질문·검색 문서·답변·정답 답변)이 됩니다.
3. **지표 계산:** 저장한 기록으로 지표를 계산해 질문별 점수와 평균을 남깁니다.
4. **기준선 비교와 원문 확인:** 프롬프트·검색 문서 수·임베딩 모델 같은 설정을 바꾸면 같은 골든셋으로 다시 계산해, 이전 결과(기준선)보다 나빠진 지표가 없는지 봅니다. 점수가 떨어진 질문은 원문을 열어 원인을 확인하고, 나빠졌으면 배포를 미룹니다.

### 8-2. 운영 중 평가: 실제 질문으로 점검

1. **로그 수집:** 실제 사용자 질문마다 질문·답변·검색 문서를 함께 저장합니다.
2. **골든셋 없이 점검:** 운영 중 질문에는 정답 답변이 없으므로 Faithfulness·Answer Relevancy(검색 쪽은 Context Relevancy)로 점검합니다. 요청이 많으면 일부를 뽑아 계산합니다.
3. **검토하고 골든셋 보강:** 점수가 낮은 답변은 사람이 원문을 확인합니다. 확인한 질문에 정답 답변을 써서 골든셋에 넣으면, 다음 배포 전 평가가 실제로 문제가 됐던 질문까지 확인합니다.

**그래서 평가 데이터는 한 번 만들고 끝나지 않습니다**

- **골든셋은 계속 자랍니다:** 운영 중에 찾은 문제 질문이 골든셋에 들어가 다음 배포 전 평가의 기준이 됩니다.
- **검색 문서까지 기록합니다:** 답변만 저장하면 검색 문제인지 답변 생성 문제인지 가릴 수 없습니다.
- **이 단원과의 관계:** 이 교안은 1~3단계를 작은 데이터로 연습했고, 교안 02는 RAG를 실제로 실행해 답변과 검색 문서를 저장한 뒤 평가합니다. 골든셋을 효율적으로 만드는 방법과 평가를 자동으로 돌리는 방법은 뒤 단원에서 다룹니다.

## 9. 🖐️ 함께 따라하기: 출산·육아 지원금 상담 답변 점검

본실습의 유연근무 매뉴얼과 다른 분야에 같은 방법을 적용합니다. 출산·육아 지원금 상담 챗봇의 답변 초안에 어떤 결함이 있는지 세 지표로 점검합니다.

자료: 행정안전부 「대한민국 공공서비스(혜택) 정보」(보조금24, 공공데이터포털, 공공누리). 실제로 신청하기 전에는 소관 기관의 안내를 확인하세요.

**아동수당 문서는 개정 전 자료입니다.** 문서에는 지급 대상이 만 8세 미만으로 적혀 있지만, 정답 답변은 현재 제도(만 9세 미만, 비수도권·인구감소지역은 금액 추가)로 썼습니다. 그래서 f2의 `근거대로` 초안은 문서대로 답했는데도 사실과 다릅니다.

**실습 진행 방법**

작성 셀은 이 함께 따라하기 절에만 있습니다.

- **부분 작성:** 셀에 남은 `...`만 주석 힌트에 맞춰 채웁니다. 나머지 코드는 제공합니다.
- `...`가 남은 채로 실행하면 에러가 나거나 빈 결과가 나옵니다. 작성 셀을 완성한 뒤 바로 아래 셀을 실행해 결과를 봅니다.
- 평가 셀이 네트워크 오류로 멈추면 그 셀만 다시 실행합니다. 결과 목록을 셀 안에서 빈 목록으로 새로 만들므로 점수가 중복으로 쌓이지 않습니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 직접 작성해 봅니다.

### 9-1. 질문과 작성 목표

- **배경:** 첫만남이용권·아동수당·육아휴직급여 질문 3개에 상담 챗봇의 답변 초안을 4개씩 준비했습니다. 질문마다 초안 3개에는 결함(금액·기한 틀림, 근거에 없는 조건 추가, 질문과 다른 안내)을 일부러 넣었습니다. 결함 종류는 본실습 변형과 하나씩 짝이 맞습니다.
- **작성:** 본실습에서 만든 `faithfulness`, `answer_relevancy`, `answer_correctness`로 초안 12개를 평가해 점수를 `practice_rows`에 쌓고, 변형별 평균 표 `practice_summary`를 만든 뒤, 초안마다 `통과`·`보류`·`문서 확인`을 판정합니다.
- **확인:** 본실습의 결함·지표 짝짓기 표처럼 결함 종류마다 어느 지표가 떨어지는지 짝짓습니다.
- **힌트:** 데이터 구조는 본실습의 `samples`와 같습니다. 질문 ID는 `question_id`에, 검색 문서는 초안마다 붙은 `retrieved_contexts`에 있습니다.

In [ ]:
# 출산·육아 지원 질문 3개를 읽습니다 (질문마다 정답 답변과 답변 초안 4개)
practice_samples = json.loads((data_dir / "따라하기" / "answer_drafts.json").read_text(encoding="utf-8"))

# 초안 12개를 한 표로 봅니다
practice_overview = []
for sample in practice_samples:
    for item in sample["responses"]:
        practice_overview.append({
            "질문 ID": sample["question_id"],
            "서비스": sample["service"],
            "질문": sample["user_input"],
            "변형": item["variant"],
            "결함": item["defect"],
        })
display(pd.DataFrame(practice_overview))

### 9-2. 초안 12개 평가와 변형별 평균

초안 하나를 평가할 때 평가용 LLM과 임베딩 모델을 합쳐 12번(Faithfulness 2번, Answer Relevancy 5번, Answer Correctness 5번) 호출합니다. 본실습 4-2처럼 `ascore`를 `await` 없이 작업 목록에 담고, 지표마다 `run_limited`로 감싸 `tqdm_asyncio.gather`로 동시에 실행합니다.

In [ ]:
# 🖐️ 함께 따라하기: 초안 12개를 세 지표로 평가해 practice_rows에 쌓습니다
pairs = []
faithfulness_tasks = []
relevancy_tasks = []
correctness_tasks = []
for sample in practice_samples:
    for item in sample["responses"]:
        pairs.append((sample, item))

        # 본실습에서 만든 세 지표의 ascore를 await 없이 작업 목록에 담으세요
        faithfulness_tasks.append(...)  # 질문·답변·검색 문서(item의 retrieved_contexts)
        relevancy_tasks.append(...)     # 질문·답변
        correctness_tasks.append(...)   # 질문·답변·정답 답변(sample의 reference)

# 지표마다 동시에 MAX_CONCURRENCY개까지 실행하고 결과를 pairs 순서대로 받습니다
faithfulness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in faithfulness_tasks], desc="Faithfulness")
relevancy_results = await tqdm_asyncio.gather(*[run_limited(task) for task in relevancy_tasks], desc="Answer Relevancy")
correctness_results = await tqdm_asyncio.gather(*[run_limited(task) for task in correctness_tasks], desc="Answer Correctness")

# 본실습과 같은 모양으로 초안마다 지표 세 행을 쌓습니다
practice_rows = []
for (sample, item), faithfulness_result, relevancy_result, correctness_result in zip(pairs, faithfulness_results, relevancy_results, correctness_results):
    for metric_name, result in [
        ("Faithfulness", faithfulness_result),
        ("Answer Relevancy", relevancy_result),
        ("Answer Correctness", correctness_result),
    ]:
        practice_rows.append({
            "질문 ID": sample["question_id"],
            "변형": item["variant"],
            "지표": metric_name,
            "점수": result.value,
        })
print("점수 행:", len(practice_rows))

In [ ]:
# 🖐️ 함께 따라하기: 변형과 지표의 조합마다 질문 3개의 점수를 평균한 표를 만드세요 (행은 변형, 열은 지표, 소수 둘째 자리)
practice_summary = ...
display(practice_summary)

**작성한 표 읽기**

- 변형마다 `근거대로`보다 크게 떨어진 지표를 찾아, 본실습 확인 퀴즈 정답 표의 같은 종류 결함(`금액_기한_틀림`은 `숫자_조건_바꿈`, `근거에_없는_조건_추가`는 `근거에_없는_내용_추가`, `질문과_다른_안내`는 `질문과_다른_이야기`)과 비교해 보세요.
- `practice_rows`에서 f2(아동수당) `근거대로` 초안의 Faithfulness와 Answer Correctness를 다른 질문의 `근거대로`와 비교해 보세요. 두 지표 중 어느 쪽이 낡은 문서 때문에 떨어지나요?

**생각해 보기**: 상담 챗봇의 답변 초안에서 금액·기한 오류를 찾아내려면 세 지표 중 무엇을 함께 봐야 할까요? Answer Relevancy만 보면 어떻게 될까요?

<details><summary>정답 보기</summary>

Faithfulness와 Answer Correctness를 함께 봅니다.

- **Faithfulness:** 검색 문서와 다른 금액·기한이 있으면 점수가 떨어집니다.
- **Answer Correctness:** 정답 답변과 다른 금액·기한이 있으면 점수가 떨어집니다.
- **Answer Relevancy만 보면:** 질문에 맞는 이야기인지만 보므로 금액이 틀린 초안도 `근거대로`와 점수가 비슷해 오류를 놓칩니다. 질문이 묻지 않은 부분을 답한 `질문과_다른_안내` 초안은 조금 낮아지지만, 같은 제도 이야기라 크게 떨어지지는 않으므로 Answer Correctness로 함께 확인합니다.

</details>

### 9-3. 출시 전 판정

점수표로 초안마다 그대로 내보낼지 정합니다. 같은 질문의 `근거대로` 초안을 기준으로, 세 지표 가운데 하나라도 0.2 이상 낮으면 `보류`, 아니면 `통과`로 표시합니다. 컴퓨터의 소수 계산은 1.0 - 0.8을 0.19999…로 내므로, 차이를 `round(…, 2)`로 반올림한 뒤 0.2와 비교합니다.

그 초안이나 기준 초안의 점수가 NaN인 지표가 있으면 차이를 계산할 수 없으므로 원문을 확인할 때까지 `보류`로 둡니다(4-2). NaN과의 비교는 늘 `False`라서, 따로 확인하지 않으면 그 초안이 조용히 `통과`로 나갑니다.

그런데 이 규칙은 기준 초안 자신의 문제는 잡지 못합니다. 기준 초안은 자기 자신과 비교하므로 차이가 늘 0이기 때문입니다. 검색 문서가 낡아 기준 초안부터 사실과 다르면, 그 질문의 Answer Correctness만 다른 질문보다 낮게 나옵니다(2-1의 사실성 오류). 그래서 기준 초안끼리도 비교합니다.

- **문서 확인:** `근거대로` 초안의 Answer Correctness가 다른 질문들의 `근거대로` 평균보다 0.2 이상 낮으면 `문서 확인`으로 표시합니다. 답변은 문서대로 만들었으니 고칠 곳은 답변이 아니라 검색 문서입니다.

In [ ]:
# 초안마다 세 지표 점수를 한 행으로 (행: 질문 ID·변형, 열: 지표)
draft_table = pd.DataFrame(practice_rows).pivot_table(index=["질문 ID", "변형"], columns="지표", values="점수")

# 질문마다 기준 초안(근거대로)의 Answer Correctness
base_correctness = {}
for (question_id, variant), row in draft_table.iterrows():
    if variant == "근거대로":
        base_correctness[question_id] = row["Answer Correctness"]

# 🖐️ 함께 따라하기: 같은 질문의 근거대로 초안보다 0.2 이상 낮은 지표가 하나라도 있으면 보류로 표시하세요
release_rows = []
for (question_id, variant), row in draft_table.iterrows():
    base = draft_table.loc[(question_id, "근거대로")]  # 같은 질문의 근거대로 초안 점수

    hold = False
    for metric in draft_table.columns:
        if pd.isna(row[metric]) or pd.isna(base[metric]):  # 이 초안이나 기준 초안의 점수가 NaN이면 비교할 수 없으므로 원문을 확인할 때까지 보류
            hold = True
        if ...:  # 이 지표 점수가 base보다 0.2 이상 낮으면 (차이를 round(…, 2)로 반올림해 비교)
            hold = True

    decision = "통과"
    if hold:
        decision = "보류"

    # 기준 초안끼리 비교: 이 질문의 근거대로 초안이 다른 질문들의 근거대로보다 Answer Correctness가 0.2 이상 낮으면 문서 확인
    if variant == "근거대로":
        others = [score for other_id, score in base_correctness.items() if other_id != question_id]
        if round(sum(others) / len(others) - row["Answer Correctness"], 2) >= 0.2:
            decision = "문서 확인"
    release_rows.append({"질문 ID": question_id, "변형": variant, "판정": decision})
display(pd.DataFrame(release_rows))

**생각해 보기**: f2(아동수당)의 `근거대로` 초안은 `문서 확인`으로 나옵니다. 답변은 문서대로 잘 만들었는데 무엇을 고쳐야 할까요? 그리고 절대 기준(예: Answer Correctness 0.5 이상이면 합격)만 썼다면 이 초안을 잡을 수 있었을까요?

<details><summary>정답 보기</summary>

검색 문서를 고쳐야 합니다. f2의 `근거대로` 초안은 개정 전 문서대로 "만 8세 미만"이라고 답해, 현재 제도(만 9세 미만, 비수도권·인구감소지역 추가 지급)로 쓴 정답 답변과 다릅니다.

- **고칠 곳은 문서입니다:** 모델은 받은 문서대로 정확히 답했으므로 답변 프롬프트나 모델을 바꿔도 나아지지 않습니다. 검색 대상 문서를 최신 안내로 바꾼 뒤 다시 평가하고, 그 전까지는 이 질문의 답변을 내보내지 않습니다.
- **절대 기준으로는 놓칩니다:** 평가용 LLM은 "만 8세 미만"을 틀린 주장(FP), "만 9세 미만"과 추가 지급을 빠뜨린 주장(FN)으로 정확히 가립니다. 하지만 맞힌 주장(금액, 지급일, 소급 조건)이 더 많고 문장도 정답 답변과 비슷해, 점수는 0.6~0.7 정도까지만 떨어집니다. 0.5 기준이면 합격입니다. 다른 질문의 `근거대로`(0.98 안팎)와 나란히 놓아야 유독 낮은 것이 보입니다.
- **정답 답변이 최신이어야 잡힙니다:** 정답 답변까지 같은 낡은 문서를 보고 썼다면 둘 다 "만 8세 미만"이라 점수가 높게 나옵니다. 골든셋의 정답 답변은 검색 대상 문서가 아니라 법령·보도자료 같은 공식 원문을 확인해 씁니다.
- **기준 초안의 점수도 흔들립니다:** 평가용 LLM의 판정은 실행마다 조금씩 달라서, 어떤 실행에서는 `근거대로` 초안의 Faithfulness가 1보다 낮게 나옵니다. 그러면 같은 질문의 결함 초안과 차이가 줄어 `통과`로 바뀔 수 있습니다. 출시 판단에 쓸 때는 같은 평가를 두세 번 실행해 평균으로 판정하고, 실행마다 판정이 바뀌는 초안은 원문을 열어 확인합니다.

</details>

## 10. 핵심 정리

- **평가 방법:** 문자열 일치와 임베딩 유사도는 정답 답변이 있어야 쓸 수 있고, LLM 기반 평가는 정답 답변이 없으면 검색 문서와 대조할 수 있습니다. RAGAS 지표 6종(검색 3·생성 3) 중 Context Precision(기본형)·Context Recall·Answer Correctness만 골든셋이 필요합니다.
- **할루시네이션:** 지표 6종이 찾는 답변 오류입니다. 검색 문서와 모순되거나 문서에 없는 내용을 덧붙인 충실성 오류는 Faithfulness로, 낡은 검색 문서를 그대로 따라 실제 사실과 다른 사실성 오류는 정답 답변과 대조하는 Answer Correctness로 찾습니다.
- **Faithfulness:** 답변의 주장 가운데 검색 문서로 뒷받침되는 주장의 비율입니다. 좋은 답변에 검색 문서에 없는 주장이 섞이면 세 지표 가운데 이 점수가 가장 크게 떨어집니다. 다만 받은 문서가 질문과 맞지 않아도 그 문서에 충실하면 높게 나옵니다.
- **Answer Relevancy:** 답변에서 거꾸로 만든 질문과 원래 질문의 유사도 평균입니다. 답변 내용이 사실인지는 확인하지 않습니다. 엇나간 답변도 0 근처까지 떨어지지 않으므로 같은 질문 안에서 변형끼리 비교합니다.
- **Answer Correctness:** 정답 답변과 비교한 사실 점수(주장 F1)와 의미 유사도를 0.75 대 0.25로 가중 평균합니다.
- **실무에서 주로 쓰는 지표:** Faithfulness와 Answer Relevancy는 정답 답변 없이 계산되어 운영 중에도 쓰고, 골든셋이 있으면 Answer Correctness와 Context Recall을 더합니다.
- **실무 절차:** 배포 전에는 골든셋으로 설정을 바꿀 때마다 같은 지표를 비교하고, 운영 중에는 실제 질문의 답변과 검색 문서를 저장해 골든셋 없이 점검한 뒤 문제 질문을 골든셋에 보탭니다.
- **읽는 법:** 점수는 평가용 LLM의 판단이라 실행마다 조금씩 달라집니다. 큰 차이만 해석하고, NaN이나 0이 나오면 원문을 열어 이유부터 확인합니다.

교안 02에서는 검색 품질 지표 3개(Context Precision·Context Recall·Context Relevancy)를 계산하고, 생성 품질 지표와 함께 RAG의 검색과 생성 중 어느 단계에서 문제가 생겼는지 진단합니다.